In [1]:
import pandas as pd

# 1. Load files
customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")

# 2. Remove duplicates
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()
reviews = reviews.drop_duplicates()

# 3. Convert order dates
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# 4. Keep delivered orders
orders = orders[orders["order_status"] == "delivered"].copy()

# 5. Join Orders + Customers
customer_churn = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

# 6. Join Reviews
customer_churn = customer_churn.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

# 7. Count orders per real customer
order_counts = (
    customer_churn
    .groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index(name="order_count")
)

# 8. Add order count
customer_churn = customer_churn.merge(
    order_counts,
    on="customer_unique_id",
    how="left"
)

# 9. Customer type
customer_churn["customer_type"] = customer_churn["order_count"].apply(
    lambda x: "Repeat Customer" if x > 1 else "One-Time Customer"
)

# 10. Delivery delay
customer_churn["delivery_delay_days"] = (
    customer_churn["order_delivered_customer_date"]
    - customer_churn["order_estimated_delivery_date"]
).dt.days

# 11. Delivery status
customer_churn["delivery_status"] = customer_churn["delivery_delay_days"].apply(
    lambda x: "Late" if x > 0 else "On Time"
)

# 12. Save cleaned dataset
customer_churn.to_csv(
    "customer_churn_analysis.csv",
    index=False
)

print("DONE")
print("Final shape:", customer_churn.shape)
print("Saved as: customer_churn_analysis.csv")

DONE
Final shape: (97007, 17)
Saved as: customer_churn_analysis.csv


In [2]:
customer_churn.shape

(97007, 17)

In [3]:
customer_churn.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,review_score,order_count,customer_type,delivery_delay_days,delivery_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,4.0,2,Repeat Customer,-8.0,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,4.0,1,One-Time Customer,-6.0,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,5.0,1,One-Time Customer,-18.0,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,5.0,1,One-Time Customer,-13.0,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,5.0,1,One-Time Customer,-10.0,On Time


In [4]:
customer_churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97007 entries, 0 to 97006
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       97007 non-null  object        
 1   customer_id                    97007 non-null  object        
 2   order_status                   97007 non-null  object        
 3   order_purchase_timestamp       97007 non-null  datetime64[ns]
 4   order_approved_at              96993 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97005 non-null  datetime64[ns]
 6   order_delivered_customer_date  96999 non-null  datetime64[ns]
 7   order_estimated_delivery_date  97007 non-null  datetime64[ns]
 8   customer_unique_id             97007 non-null  object        
 9   customer_zip_code_prefix       97007 non-null  int64         
 10  customer_city                  97007 non-null  object        
 11  customer_state 